<a href="https://colab.research.google.com/github/DaisyXUUUUU/CIS-5450-Project/blob/main/notebooks/01c_reproducibility_fresh_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1c — Reproducibility test: fresh download → the same 796,989 trips?

**Owner:** Ziyue Xu (Daisy)

Question: if someone else starts from the official links, with the same seed and the same rules, do they get **exactly the
same** analysis table we use (`trips_clean_v1.parquet`, 796,989 rows; and the 6 extra v2 columns)?

What this notebook does:

1. Downloads the April–June 2026 yellow taxi Parquet files, the zone lookup, and the hourly weather **fresh from the
   original URLs** into the Colab disk (`/content/fresh/`). Nothing is written to Google Drive.
2. Prints file sizes and SHA-256 checksums and compares them with the copies in `MyDrive/.../data/raw/` (if present).
3. Re-runs the original pipeline from `01_data_pipeline.ipynb` (DuckDB hash sample with `SEED = 5450`, the 8 cleaning
   rules, zone and weather joins, features, target, split) and `01b_add_features_v2.ipynb` (6 v2 columns), using the same code.
4. Compares every stage with the recorded numbers and with the fixed files on Drive, and prints **True / False** for each.

Google Drive is only **read** (to compare). The existing notebooks and data files are not changed.

Note: the sample uses DuckDB's `hash()`, which can change between DuckDB versions. The original sample was drawn with
**DuckDB 1.3.2**; the version used here is printed below.

## 0. Setup

In [ ]:
import os, sys, time, hashlib, subprocess, tempfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

try:
    import duckdb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.3.2"], check=True)
    import duckdb
import numpy as np
import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("duckdb", duckdb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
if duckdb.__version__ != "1.3.2":
    print("WARNING: the original sample was drawn with DuckDB 1.3.2; hash() may give a different sample in this version.")

In [ ]:
# ---- design (identical to 01_data_pipeline.ipynb) -----------------------------------
SEED = 5450
MONTHS = ["2026-04", "2026-05", "2026-06"]
N_PER_MONTH = 300_000
STUDY_START, STUDY_END = pd.Timestamp("2026-04-01"), pd.Timestamp("2026-07-01")
TRAIN_END, VAL_END = pd.Timestamp("2026-05-16"), pd.Timestamp("2026-06-01")
HIGH_TIP_THRESHOLD, SENSITIVITY_THRESHOLDS = 0.25, [0.20, 0.30]
RAIN_MM = 0.1
NYC_LAT, NYC_LON = 40.7128, -74.0060
JFK, LGA, EWR = 132, 138, 1
UNKNOWN_ZONES = [264, 265]

# ---- where things are ----------------------------------------------------------------
FRESH_DIR = Path("/content/fresh") if IN_COLAB else Path(tempfile.gettempdir()) / "cis5450_fresh"
FRESH_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path(os.environ.get("CIS5450_DATA_DIR", "/content/drive/MyDrive/CIS5450_project/data"))
DRIVE_RAW, DRIVE_CLEAN = DATA_DIR / "raw", DATA_DIR / "clean"      # read only
V1_PATH, V2_PATH = DRIVE_CLEAN / "trips_clean_v1.parquet", DRIVE_CLEAN / "trips_clean_v2.parquet"
DRIVE_SAMPLE = DRIVE_RAW / "sample_card_trips.parquet"

TLC_BASE = os.environ.get("CIS5450_TLC_BASE", "https://d37ci6vzurychx.cloudfront.net")
SOURCES = {f"yellow_tripdata_{m}.parquet": f"{TLC_BASE}/trip-data/yellow_tripdata_{m}.parquet" for m in MONTHS}
SOURCES["taxi_zone_lookup.csv"] = f"{TLC_BASE}/misc/taxi_zone_lookup.csv"
FRESH_RAW = {m: FRESH_DIR / f"yellow_tripdata_{m}.parquet" for m in MONTHS}

# ---- recorded results of the original run (2026-11-03) ----------------------------------
EXPECTED = {
    "raw_rows":   {"2026-04": 3_831_240, "2026-05": 4_090_836, "2026-06": 3_837_248},
    "card_trips": {"2026-04": 2_635_749, "2026-05": 2_727_585, "2026-06": 2_432_871},
    "funnel_rows_after": [900_000, 825_235, 825_233, 825_232, 804_723, 802_836, 798_194, 798_194, 796_989],
    "final_rows": 796_989,
    "split_rows": {"train": 395_144, "val": 127_349, "test": 274_496},
}
RESULTS = {}      # stage -> True / False / None (None = could not be checked)
def record(stage, ok):
    RESULTS[stage] = ok
    print(f"[{'PASS' if ok else ('n/a ' if ok is None else 'FAIL')}] {stage}")

print("fresh files ->", FRESH_DIR)
print("Drive (read only) ->", DATA_DIR, "| v1 exists:", V1_PATH.exists(), "| v2 exists:", V2_PATH.exists())

## 1. Fresh download from the official links

Files go to the Colab disk only. In a new Colab session `/content/fresh/` is empty, so everything is downloaded again.

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

def download(url, dest):
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[already downloaded in this session] {dest.name}")
        return
    t0 = time.time()
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + ".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
        tmp.rename(dest)
    print(f"[downloaded] {dest.name}  {dest.stat().st_size / 1e6:.1f} MB in {time.time() - t0:.0f} s")

for name, url in SOURCES.items():
    download(url, FRESH_DIR / name)

# Weather: same Open-Meteo request as the original pipeline
FRESH_WEATHER = FRESH_DIR / "weather_hourly_nyc.csv"
if not FRESH_WEATHER.exists():
    resp = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=120, params={
        "latitude": NYC_LAT, "longitude": NYC_LON, "start_date": "2026-04-01", "end_date": "2026-06-30",
        "hourly": "temperature_2m,precipitation", "timezone": "America/New_York"})
    resp.raise_for_status()
    pd.DataFrame(resp.json()["hourly"]).to_csv(FRESH_WEATHER, index=False)
    print("[downloaded] weather_hourly_nyc.csv")
print("Download date (UTC):", pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M"))

### Are the downloaded files byte-identical to the copies on Drive?

If TLC re-published a file since 2026-11-03, the checksum changes; the row-level checks below still tell us whether the
trips we use are the same.

In [ ]:
rows = []
for name in list(SOURCES) + ["weather_hourly_nyc.csv"]:
    fresh, old = FRESH_DIR / name, DRIVE_RAW / name
    row = {"file": name, "fresh_MB": round(fresh.stat().st_size / 1e6, 2), "fresh_sha256": sha256(fresh)[:16] + "..."}
    if old.exists():
        row["drive_MB"] = round(old.stat().st_size / 1e6, 2)
        row["same_bytes_as_drive"] = row["fresh_sha256"] == sha256(old)[:16] + "..." and fresh.stat().st_size == old.stat().st_size
    else:
        row["drive_MB"], row["same_bytes_as_drive"] = None, None
    rows.append(row)
checksums = pd.DataFrame(rows)
display(checksums)
tlc_files = [f"yellow_tripdata_{m}.parquet" for m in MONTHS]
same = checksums.set_index("file").loc[tlc_files, "same_bytes_as_drive"]
record("1. TLC trip files byte-identical to the Drive copies", None if same.isna().any() else bool(same.all()))

## 2. Raw row counts and card trips

In [ ]:
con = duckdb.connect()
raw = []
for m in MONTHS:
    n_rows, n_card = con.execute(f"SELECT count(*), count(*) FILTER (WHERE payment_type = 1) FROM read_parquet('{FRESH_RAW[m]}')").fetchone()
    raw.append({"month": m, "rows": n_rows, "expected_rows": EXPECTED["raw_rows"][m],
                "card_trips": n_card, "expected_card_trips": EXPECTED["card_trips"][m]})
raw = pd.DataFrame(raw)
display(raw)
record("2a. raw rows per month match", bool((raw["rows"] == raw["expected_rows"]).all()))
record("2b. card trips per month match", bool((raw["card_trips"] == raw["expected_card_trips"]).all()))

## 3. Hash sample (same SQL as the original)

In [ ]:
parts = []
for m in MONTHS:
    part = con.execute(f'''
        SELECT * EXCLUDE (file_row_number)
        FROM read_parquet('{FRESH_RAW[m]}', file_row_number = true)
        WHERE payment_type = 1
        ORDER BY hash(file_row_number, {SEED})
        LIMIT {N_PER_MONTH}
    ''').df()
    part["source_month"] = m
    parts.append(part)
sample = pd.concat(parts, ignore_index=True)
for c in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    sample[c] = pd.to_datetime(sample[c])
print(f"Fresh sample: {len(sample):,} rows")
print(sample["source_month"].value_counts().sort_index().to_string())
record("3a. 300,000 card trips per month", bool(sample["source_month"].value_counts().eq(N_PER_MONTH).all() and len(sample) == 3 * N_PER_MONTH))

KEY_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "VendorID",
            "trip_distance", "fare_amount", "tip_amount", "total_amount", "extra", "tolls_amount"]

def key_frame(d):
    """Order-free fingerprint of a set of trips."""
    k = d[KEY_COLS].copy()
    for c in KEY_COLS[:2]:
        k[c] = pd.to_datetime(k[c]).astype("datetime64[ns]")
    for c in KEY_COLS[2:]:
        k[c] = k[c].astype("float64")
    return k.sort_values(KEY_COLS).reset_index(drop=True)

if DRIVE_SAMPLE.exists():
    saved = pd.read_parquet(DRIVE_SAMPLE)
    record("3b. same 900,000 trips as the saved sample on Drive", key_frame(sample).equals(key_frame(saved)))
    del saved
else:
    record("3b. same 900,000 trips as the saved sample on Drive", None)

## 4. Cleaning (same 8 rules, same order)

In [ ]:
df = sample.drop(columns=["request_source"], errors="ignore").copy()
df["duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

RULES = [
    ("1. tip_amount > 0",                     lambda d: d["tip_amount"] > 0),
    ("2. fare_amount > 0",                    lambda d: d["fare_amount"] > 0),
    ("3. pickup within Apr 1 - Jun 30, 2026", lambda d: d["tpep_pickup_datetime"].between(STUDY_START, STUDY_END, inclusive="left")),
    ("4. 1 <= duration <= 180 min",           lambda d: d["duration_min"].between(1, 180)),
    ("5. 0 < trip_distance <= 100 mi",        lambda d: (d["trip_distance"] > 0) & (d["trip_distance"] <= 100)),
    ("6. PU/DO zone not 264/265",             lambda d: ~d["PULocationID"].isin(UNKNOWN_ZONES) & ~d["DOLocationID"].isin(UNKNOWN_ZONES)),
    ("7. RatecodeID != 99",                   lambda d: d["RatecodeID"].ne(99) | d["RatecodeID"].isna()),
    ("8. tip_pct <= 100%",                    lambda d: d["tip_amount"] / d["fare_amount"] <= 1.0),
]
funnel = [{"step": "0. sampled card trips", "rows_after": len(df)}]
for name, rule in RULES:
    df = df[rule(df).fillna(False).astype(bool)]
    funnel.append({"step": name, "rows_after": len(df)})
funnel = pd.DataFrame(funnel)
funnel["expected_rows_after"] = EXPECTED["funnel_rows_after"]
funnel["match"] = funnel["rows_after"] == funnel["expected_rows_after"]
df = df.reset_index(drop=True)
df.loc[df["passenger_count"] == 0, "passenger_count"] = np.nan
display(funnel)
record("4a. every funnel step matches the recorded row count", bool(funnel["match"].all()))
record("4b. final row count = 796,989", len(df) == EXPECTED["final_rows"])

## 5. Joins, features, target, split (same code as the original)

In [ ]:
zones = pd.read_csv(FRESH_DIR / "taxi_zone_lookup.csv")
con.register("trips", df)
con.register("zones", zones)
n_before = len(df)
df = con.execute('''
    SELECT t.*,
           pz.Borough AS PU_Borough, pz.Zone AS PU_Zone, pz.service_zone AS PU_service_zone,
           dz.Borough AS DO_Borough, dz.Zone AS DO_Zone, dz.service_zone AS DO_service_zone
    FROM trips AS t
    LEFT JOIN zones AS pz ON t.PULocationID = pz.LocationID
    LEFT JOIN zones AS dz ON t.DOLocationID = dz.LocationID
''').df()
assert len(df) == n_before

weather = pd.read_csv(FRESH_WEATHER, parse_dates=["time"]).rename(
    columns={"time": "weather_hour", "temperature_2m": "temp_c", "precipitation": "precip_mm"})
df["pickup_hour_ts"] = df["tpep_pickup_datetime"].dt.floor("h")
con.register("trips2", df)
con.register("weather", weather)
df = con.execute('''
    SELECT t.*, w.temp_c, w.precip_mm
    FROM trips2 AS t
    LEFT JOIN weather AS w ON t.pickup_hour_ts = w.weather_hour
''').df()
assert len(df) == n_before
df["is_rain"] = (df["precip_mm"] >= RAIN_MM).astype("Int8").where(df["precip_mm"].notna())

pu = df["tpep_pickup_datetime"]
df["pickup_hour"]   = pu.dt.hour.astype("int8")
df["pickup_dow"]    = pu.dt.dayofweek.astype("int8")
df["is_weekend"]    = (df["pickup_dow"] >= 5).astype("int8")
df["is_late_night"] = (df["pickup_hour"] <= 5).astype("int8")
df["avg_speed_mph"] = 60 * df["trip_distance"] / df["duration_min"]
df["is_airport"]    = (df["PULocationID"].isin([JFK, LGA]) | df["DOLocationID"].isin([JFK, LGA])).astype("int8")
df["is_newark"]     = (df["PULocationID"].eq(EWR) | df["DOLocationID"].eq(EWR)).astype("int8")
df["same_borough"]  = (df["PU_Borough"] == df["DO_Borough"]).astype("int8")
df["has_cbd_fee"]   = (df["cbd_congestion_fee"].fillna(0) > 0).astype("int8")
df["is_short_trip"] = (df["trip_distance"] <= 2).astype("int8")

df["tip_pct"] = df["tip_amount"] / df["fare_amount"]
df["high_tip"] = (df["tip_pct"] >= HIGH_TIP_THRESHOLD).astype("int8")
for t in SENSITIVITY_THRESHOLDS:
    df[f"high_tip_{int(t * 100)}"] = (df["tip_pct"] >= t).astype("int8")
df["pickup_date"] = df["tpep_pickup_datetime"].dt.normalize()
df["split"] = np.select([df["tpep_pickup_datetime"] < TRAIN_END, df["tpep_pickup_datetime"] < VAL_END],
                        ["train", "val"], default="test")
df = df.sort_values("tpep_pickup_datetime").reset_index(drop=True)
rebuilt_v1 = df
print("Rebuilt v1:", rebuilt_v1.shape)
split_rows = rebuilt_v1["split"].value_counts().to_dict()
print("split rows:", split_rows)
record("5. split rows = 395,144 / 127,349 / 274,496", split_rows == EXPECTED["split_rows"])

## 6. Compare with the fixed `trips_clean_v1.parquet` on Drive

Both tables are sorted the same way (by many columns, so the order is unique), then every column is compared value by
value. Weather columns are reported separately: Open-Meteo can revise its historical data, which would change
`temp_c` / `precip_mm` without changing which trips are in the table.

In [ ]:
SORT_COLS = KEY_COLS + ["passenger_count", "RatecodeID", "mta_tax", "improvement_surcharge",
                        "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]
WEATHER_COLS = ["temp_c", "precip_mm", "is_rain"]

def canonical(d):
    d = d.copy()
    for c in d.columns:
        if pd.api.types.is_datetime64_any_dtype(d[c]):
            d[c] = d[c].astype("datetime64[ns]")
    return d.sort_values(SORT_COLS, kind="mergesort", na_position="last").reset_index(drop=True)

def compare_columns(a, b):
    A, B = canonical(a), canonical(b)
    out = {}
    for c in a.columns:
        if c not in B:
            out[c] = False
            continue
        try:
            pd.testing.assert_series_equal(A[c], B[c], check_dtype=False, check_exact=True, check_names=False)
            out[c] = True
        except AssertionError:
            out[c] = False
    return pd.Series(out, name="identical")

if V1_PATH.exists():
    fixed_v1 = pd.read_parquet(V1_PATH)
    print("Fixed v1:", fixed_v1.shape, "| rebuilt v1:", rebuilt_v1.shape)
    record("6a. same shape as fixed v1 (796,989 x 48)", fixed_v1.shape == rebuilt_v1.shape)
    record("6b. same column names and order", list(fixed_v1.columns) == list(rebuilt_v1.columns))
    record("6c. exactly the same set of trips (key columns)", key_frame(rebuilt_v1).equals(key_frame(fixed_v1)))
    col_check = compare_columns(rebuilt_v1, fixed_v1)
    bad = col_check[~col_check].index.tolist()
    print("Columns that differ:", bad if bad else "none")
    record("6d. every non-weather column identical, value by value", all(col_check.drop(WEATHER_COLS, errors="ignore")))
    record("6e. weather columns identical (Open-Meteo data unchanged)", all(col_check.reindex(WEATHER_COLS).fillna(False)))
else:
    print("Fixed v1 not found on Drive; cannot compare.")
    for s in ["6a", "6b", "6c", "6d", "6e"]:
        record(f"{s}. compare with fixed v1", None)

## 7. v2 columns (same code as `01b_add_features_v2.ipynb`) vs the fixed `trips_clean_v2.parquet`

In [ ]:
v2 = rebuilt_v1.copy()
pu = v2["tpep_pickup_datetime"]
hour, dow = pu.dt.hour, pu.dt.dayofweek
v2["is_rush_hour"] = ((dow <= 4) & hour.between(16, 19)).astype("int8")
v2["is_night"] = ((hour >= 20) | (hour <= 5)).astype("int8")
bill = (v2["total_amount"] - v2["tip_amount"]).round(2)
v2["tip_pct_bill"] = np.where(bill > 0, v2["tip_amount"] / bill, np.nan)
v2["high_tip_bill"] = (v2["tip_amount"] > (0.20 * bill).round(2) + 0.015).astype("int8")
v2.loc[bill <= 0, "high_tip_bill"] = 0
for t in (0.25, 0.30):
    col = f"high_tip_bill_{int(t * 100)}"
    v2[col] = (v2["tip_amount"] > (t * bill).round(2) + 0.015).astype("int8")
    v2.loc[bill <= 0, col] = 0

if V2_PATH.exists():
    fixed_v2 = pd.read_parquet(V2_PATH)
    record("7a. same shape as fixed v2 (796,989 x 54)", fixed_v2.shape == v2.shape)
    new_cols = ["is_rush_hour", "is_night", "tip_pct_bill", "high_tip_bill", "high_tip_bill_25", "high_tip_bill_30"]
    col_check_v2 = compare_columns(v2, fixed_v2)
    print(col_check_v2.reindex(new_cols).to_string())
    record("7b. the 6 v2 columns identical, value by value", bool(col_check_v2.reindex(new_cols).fillna(False).all()))
else:
    record("7a. compare with fixed v2", None)
    record("7b. compare with fixed v2", None)

## 8. Result

In [ ]:
summary = pd.Series(RESULTS, name="result").map({True: "PASS", False: "FAIL", None: "n/a"}).to_frame()
display(summary)
core = ["3a. 300,000 card trips per month", "4a. every funnel step matches the recorded row count",
        "4b. final row count = 796,989", "6c. exactly the same set of trips (key columns)",
        "6d. every non-weather column identical, value by value"]
if all(RESULTS.get(k) is True for k in core):
    print("REPRODUCED: a fresh download with SEED = 5450 gives exactly the same 796,989 trips and values as the fixed table.")
else:
    print("NOT REPRODUCED: see the FAIL rows above.")
con.close()